In [1]:
import sys
sys.path.insert(1, '../../../scripts/')
from preprocess import preprocess
from preprocess import correct_inputs 

from utils import utils_2
from utils import functions as func
from tqdm import tqdm
from utils import parameters as params
import copy
from utils import metabolites as metab
import pickle
import sympy

import h5py
import numpy as np
import pandas as pd
import pickle

lp_path = '/data2/hratch/human_me/test_lp/'

ERROR:cobra.io.sbml:No objective coefficients in model. Unclear what should be optimized


In [2]:
counter = 4
base = 3

mu_val = 1e-9

In [3]:
from expression import build_me_model
tme, builder = build_me_model.build_me(minimal_proteome = True, compress_mrna = False, 
                                                model_id = 'toy_me_model')
    
    

# counter = 2
# with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

# with open(lp_path + 'notworking_version.pickle', 'rb') as handle:
#     tme = pickle.load(handle)

Generate ubiquitin reactions for proteasomal degrdation
Generate ribosome


../../../scripts/expression/protein_expression/ubiquitin.py:34 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
../../../scripts/expression/protein_expression/ubiquitin.py:64 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
../../../scripts/expression/gene_information.py:114 UserWarning: HGNC:10368: The letter X is in the protein sequence. Replacing with a random amino acid
  1%|          | 6/591 [00:00<00:11, 51.20it/s]

Generate protein expression reactions for metabolic enzymes and non-machinery


100%|██████████| 591/591 [00:13<00:00, 44.89it/s]


Generate protein expression reactions for expression module enzymes


...

A total of 1897 reactions were dropped when forming a minimal proteome
Add machinery to metabolic module reactions


  0%|          | 33/10659 [00:00<00:32, 322.38it/s]

Add machinery to expression module reactions


100%|██████████| 10659/10659 [00:44<00:00, 238.31it/s]


Generate ME-Model
Time to build: 4.424628734588623 minutes


In [4]:
sln, stat, _ = tme.solve_lp(mu_val = mu_val)

fn = '/data2/hratch/human_me/test_lp/S_matrix.h5'
if stat == 0:
    S = tme.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'pandas')
    S.to_hdf(fn, key = str(counter), mode = 'a')
    print('Last saved file: {}'.format(counter))
else:
    infeasible_reactions = tme.infeasible_reactions(mu_val = mu_val, sln = sln)
    raise ValueError('Model did not solve')

Getting MINOS parameters...
Done in 138.896 seconds with status 0


/home/hratch/anaconda3/envs/CD8T_RA/lib/python3.6/site-packages/tables/path.py:155 NaturalNameWarning: object name is not a valid Python identifier: '3'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though


Last saved file: 3


In [ ]:
def save_me_model(me_model, counter):
    print('Success, please update git')
    lp_path = '/data2/hratch/human_me/test_lp/'
    with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'wb') as handle:
        pickle.dump(me_model, handle)

def get_changes(S_1, S_0):
    mismatch = np.argwhere(np.not_equal(S_0.values, S_1.values))
    am = S_1.index.tolist()
    mm = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm:
        mm[m]['reactions'] = sorted(set([t[1] for t in mismatch if t[0] == m]))

    am, rm = S_1.index.tolist(), S_1.columns.tolist()
    mm_2 = {m: {'id': am[m]} for m in sorted(set([t[0] for t in mismatch]))}
    for m in mm_2:
        mm_2[m]['reactions'] = sorted(set(['_'.join(rm[t[1]].split('_')[1:]) if 'HGNC' in rm[t[1]] else rm[t[1]] for t in mismatch if t[0] == m]))
    
    return mismatch, mm, mm_2

In [5]:
# S_1 = pd.read_hdf(fn, key = str(counter))
# S_0 = pd.read_hdf(fn, key = str(base))

if not S_0.equals(S_1):
    indeces = True
    if indeces:
        if S_1.shape != S_0.shape:
            print('Dimensions are not the same')
            indeces = False
        if len(set(S_1.columns).difference(S_0.columns)) > 0:
            print('Columns are not the same')
            indeces = False
        if len(set(S_1.index).difference(S_0.index)) > 0:
            print('Rows are not the same')
            indeces = False
    if indeces:
        S_1 = S_1.loc[S_0.index, S_0.columns]
        if not S_0.equals(S_1):
            mismatch, mm, mm_2 = get_changes(S_1, S_0)
            print('Dataframes are not equal due to stoichiometric values mismatch, will not save model')
        else:
            save_me_model(tme, counter)
    else:
        print('Dataframes are not equal due to column/row label mismatch, will not save model')
else:
    save_me_model(tme, counter)

Success, please update git


In [15]:
mm.keys()

dict_keys([774, 1992, 1994, 1996, 2667, 2668, 10048])

In [40]:
m_idx = 10048
mm_2[m_idx]

{'id': 'biomass_unmodeled_protein',
 'reactions': ['folded_protein_c_DEUBIQUITINATIONc']}

In [41]:
S_1.iloc[m_idx, mm[m_idx]['reactions']]

HGNC:DUMMY_folded_protein_c_DEUBIQUITINATIONc    0.018015
Name: biomass_unmodeled_protein, dtype: float64

In [42]:
S_0.iloc[m_idx, mm[m_idx]['reactions']]

HGNC:DUMMY_folded_protein_c_DEUBIQUITINATIONc    0.018015
Name: biomass_unmodeled_protein, dtype: float64

In [43]:
test = S_1.iloc[m_idx, mm[m_idx]['reactions']]  - S_0.iloc[m_idx, mm[m_idx]['reactions']]
test[abs(test) > 1e-8]

Series([], Name: biomass_unmodeled_protein, dtype: float64)

# testing ubiquitin cleavage

In [37]:
def binary_search(tme, bm_min=-17.111457840000003, bm_max=-0.1, accuracy=0.1):
    feasible_mu = [bm_min]
    infeasible_mu = [bm_max]
    def replace_biomass(biomass_val):
        new_reactions = [r.copy() for r in tme.reactions]
        r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
        r_.add_metabolites({tme.metabolites.get_by_id('biomass_protein'): biomass_val}, 
                         combine = False)
        if len(r_.check_mass_balance()) == 0:
            test_me = func.ME_Model('test')
            test_me.add_reactions(new_reactions)
            print('Begin solve')
            sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)
        if stat0.max() == 0:
            feasible_mu.append(biomass_val)
            return True, sln0, stat0
        elif stat0.max() == 1:
            infeasible_mu.append(biomass_val)
            return False, sln0, stat0
        else:
            raise ValueError('Something went wrong')
    
    while (abs(infeasible_mu[-1] - feasible_mu[-1])) > accuracy:
        print('Current infeasible: {}'.format(infeasible_mu[-1]))
        print('Current feasible: {}'.format(feasible_mu[-1]))
        bool_, sln,stat = replace_biomass((infeasible_mu[-1] + feasible_mu[-1]) * 0.5)
        print('--------------')
    
    return sln, stat, feasible_mu, infeasible_mu
    
    


In [6]:
biomass_val = 0.018

new_reactions = [r.copy() for r in tqdm(tme.reactions)]
r_ = [r for r in new_reactions if r.id == 'HGNC:12458_UBIQUITIN_CLEAVAGEc'][0]
r_.add_metabolites({tme.metabolites.get_by_id('biomass_protein'): biomass_val}, 
                 combine = False)
if len(r_.check_mass_balance()) == 0:
    test_me = func.ME_Model('test')
    test_me.add_reactions(new_reactions)
    print('Begin solve')
    sln0, stat0, _ = test_me.solve_lp(mu_val = 1e-9)

Begin solve
Getting MINOS parameters...
Done in 172.355 seconds with status 1


In [7]:
test = infeasible_reactions = tme.infeasible_reactions(mu_val = mu_val, sln = sln0)



In [9]:
len(test)

1519

In [ ]:
len()